[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/purple/notebooks/purple_baseline_models.ipynb)

# Training baseline models for HIV-1 activity

**Purple group · HIV**

This notebook trains the group's baseline machine learning models on the HIV-1 data curated
in `purple_data_curation`: one that says whether a molecule is active, and one that predicts
how active it is. Each is tested twice, on a random split and on a harder scaffold split, and
the two final models are saved so the group can use them on new molecules.

## What you will do

- Look at the data and clean it: drop the handful of molecules too large to model sensibly.
- Turn every molecule into a fingerprint, a row of numbers a model can read.
- Train a random forest to predict active or inactive, and score it on a random split and on
  a scaffold split.
- Do the same for a random forest that predicts potency itself.
- Train the two final models on all the data and download them.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "purple"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the curated data

The curation notebook, `purple_data_curation`, produced two tables, and the group uploaded them to its Drive
folder. They are copied into `data/` in this repository, so they are already here.

- `hiv1_curated.csv` has one row per molecule and an `activity` label: 1 for active,
  0 for inactive, with the line drawn at 1 uM. We use it for **classification**.
- `hiv1_regression.csv` keeps only the molecules with a real measured value, in the
  `pactivity` column. We use it for **regression**.

In [ ]:
import numpy as np
import pandas as pd
import stylia
from scripts import modelling

RANDOM_SEED = 42
THRESHOLD = 6.0  # pActivity 6 is the 1 uM cutoff used to label the molecules

curated = pd.read_csv("data/hiv1_curated.csv")
regression = pd.read_csv("data/hiv1_regression.csv")
print(f"classification set: {len(curated):,} molecules")
print(f"regression set:     {len(regression):,} molecules")

The columns we need are the `smiles` (the molecule), `activity` (the label) and
`pactivity` (the potency, where higher means more potent).

In [ ]:
curated[["inchikey", "smiles", "pactivity", "activity"]].head()

## 2. Look at the data

Before training anything, look at what the model will learn from. A model can only be
as good as its data, and a few plots often reveal problems that no metric will show
later: too few actives, a strange spread of values, or molecules that are very
different from the rest.

We start with the **class balance**: how many molecules are active and how many are
inactive.

In [ ]:
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

counts = curated["activity"].value_counts().rename({0: "inactive", 1: "active"})
counts = counts.reindex(["inactive", "active"])  # the order the colours below assume

fig, axs = stylia.create_figure(1, 1, width=0.3, height=0.4)
ax = axs.next()
ax.bar(counts.index, counts.values, color=[nc.purple, nc.mint])
stylia.label(ax, xlabel="", ylabel="Molecules",
             title=f"{counts['active'] / counts.sum():.0%} of the molecules are active")

The two classes are almost the same size. That is good news: when one class is much
rarer than the other (class imbalance), a model can look accurate just by always
predicting the common class. Here that trap is not a worry, but it will come back if
the cutoff changes.

> **Note:** The project plan lists class imbalance as a risk. With a 1 uM cutoff it
> is not a problem for this data, but a stricter cutoff such as 0.1 uM would leave far
> fewer actives. The cutoff is set in the curation notebook, section 7.

Next, the **potency distribution** of the regression set. The dashed line is the
cutoff: everything to its right is active.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(regression["pactivity"], bins=60, range=(3, 11.5), color=nc.purple)
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--")
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title=f"Measured potency (median {regression['pactivity'].median():.2f})")

Most molecules sit between pActivity 4 and 9, that is between 100 uM and 1 nM. A
regression model has to predict a number somewhere on this scale.

Size can also give a model an easy shortcut. If actives were simply bigger than
inactives, a model could learn "big means active" and nothing about chemistry. We
compute the **molecular weight** of every molecule to check.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

curated["mw"] = [Descriptors.MolWt(Chem.MolFromSmiles(smi)) for smi in curated["smiles"]]
curated.groupby("activity")["mw"].describe().round(0)

The histogram shows the two classes side by side.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for label, color, name in [(0, nc.purple, "inactive"), (1, nc.mint, "active")]:
    weights = curated.loc[curated["activity"] == label, "mw"]
    ax.hist(weights, bins=60, range=(0, 1200), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{name} (median {weights.median():.0f})")
ax.legend()
stylia.label(ax, xlabel="Molecular weight (g/mol)", ylabel="Molecules",
             title="Molecular weight of actives and inactives")

Actives are somewhat heavier than inactives: their median weight is about 80 g/mol
higher. So size alone carries some information about activity, and a model may lean
on it. That is not wrong, but it is worth remembering when the model later scores
natural products, which are often large.

Last, the **chemical series**. Medicinal chemists usually make many variations of the
same core structure. That core is called the **scaffold** (or Bemis-Murcko scaffold):
the rings of a molecule and the chains that connect them, with every side group
removed. We will need the scaffolds in section 6, so we compute them now. This takes
about a minute.

In [ ]:
curated["scaffold"] = modelling.murcko_scaffolds(curated["smiles"])
series = curated["scaffold"].value_counts()
print(f"{len(curated):,} molecules share {len(series):,} scaffolds")
print(f"{(series == 1).sum():,} scaffolds appear only once")
series.head(5).to_frame("molecules")

Some scaffolds hold hundreds of molecules. These are the series that chemists
explored most. Keep this in mind: two molecules from the same series are often
almost identical, and that matters when we test a model.

The most common scaffold, five fused rings, is the skeleton of **betulinic acid**, a
natural product found in the bark of birch and other trees. Its derivatives, such as
bevirimat, block the last step of HIV maturation. So natural products are already
part of this dataset. (`c1ccccc1` is a plain benzene ring: molecules with a single
ring all share it.)

> **Exercise:** Paste a few of the top scaffolds into a structure viewer (for example
> https://molview.org). Then count how many molecules of the betulinic acid series are
> active, using `curated[curated["scaffold"] == series.index[0]]`.

## 3. Clean the molecules

The curation notebook already removed salts and duplicates, but it kept everything that had
a measurement, and a few of those are not really small molecules at all. The heaviest entry
in this table weighs about 9,500 g/mol: that is a protein, not a drug candidate.

Molecules that large are a problem for two reasons. A fingerprint describes them no better
than it describes a peptide ten times smaller, and the group is looking for **natural
products taken by mouth**, which essentially never weigh more than about 1,000 g/mol. So we
drop everything above that line. It costs a small percentage of the data and removes a part
of chemical space the model was never going to be used on.

In [ ]:
too_big = curated["mw"] > 1000
print(f"{too_big.sum():,} molecules above 1,000 g/mol removed "
      f"({too_big.mean():.1%} of the data, {curated.loc[too_big, 'activity'].mean():.0%} of them active)")

curated = curated[~too_big].reset_index(drop=True)
regression = regression[regression["inchikey"].isin(set(curated["inchikey"]))].reset_index(drop=True)
print(f"{len(curated):,} molecules kept for classification, {len(regression):,} for regression")

The molecules we dropped were about as often active as the ones we kept, so this
filter does not tip the balance between the two classes. That is worth checking whenever you
remove data: a filter that removed mostly actives would quietly change the problem.

> **Exercise:** Change the 1,000 above to 800 and rerun from here. How many more molecules
> go, and do the scores at the end of the notebook move at all?

## 4. Turn molecules into numbers

A model cannot read a SMILES string. It needs every molecule written as the same fixed
list of numbers, called **features**. Turning molecules into features is called
**featurisation**.

We use a **Morgan fingerprint** (also called ECFP4). For every atom, it looks at the
small fragment around it, up to two bonds away, and switches on one of 2,048 bits for
that fragment. Two molecules that share many fragments share many bits, so similar
molecules get similar fingerprints.

In [ ]:
X_class = modelling.morgan_fingerprints(curated["smiles"], radius=2, n_bits=2048)
y_class = curated["activity"].values
print(f"feature matrix: {X_class.shape[0]:,} molecules x {X_class.shape[1]:,} bits")
print(f"on average {X_class.sum(axis=1).mean():.0f} bits are switched on per molecule")

The regression molecules are all in the classification table already, so instead of
computing their fingerprints again we pick out their rows. We do the same for their
scaffolds.

In [ ]:
rows = pd.Series(range(len(curated)), index=curated["inchikey"])[regression["inchikey"]]
X_reg = X_class[rows.values]
y_reg = regression["pactivity"].values
regression["scaffold"] = curated["scaffold"].values[rows.values]
print(f"feature matrix: {X_reg.shape[0]:,} molecules x {X_reg.shape[1]:,} bits")

> **Note:** Most bits are 0 for any given molecule: only about 3% are switched on.
> That is normal for fingerprints, and random forests handle it well.

## 5. Predict active or inactive

The first question is the simpler one: given a molecule, is it active against HIV-1 or not.

### 5.1 A first model on a random split

To know whether a model works we must test it on molecules it has **never seen**. So we hold
back 20% of the molecules as a **test set** and train only on the other 80%, the **training
set**. The split is stratified, meaning both parts keep the same share of actives.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_class, y_class, test_size=0.2, stratify=y_class, random_state=RANDOM_SEED)
print(f"training set: {len(y_train):,} molecules, {y_train.mean():.1%} active")
print(f"test set:     {len(y_test):,} molecules, {y_test.mean():.1%} active")

A **random forest** is a collection of decision trees. Each tree asks a series of
yes-or-no questions about the bits ("does the molecule contain this fragment?"), and
the forest averages the answers of all its trees. It is fast, needs little tuning and
is a common first choice in cheminformatics.

`class_weight="balanced"` tells the model to care equally about both classes even when
one is rarer. It makes little difference here, but it is a safe habit.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                    n_jobs=-1, random_state=RANDOM_SEED)
classifier.fit(X_train, y_train)
proba = classifier.predict_proba(X_test)[:, 1]
print(f"predicted the probability of being active for {len(proba):,} test molecules")

The model gives each test molecule a **probability** of being active, between 0 and 1.
We call a molecule active when that probability is 0.5 or more, and compare with the
real labels. The metrics mean:

- **ROC-AUC**: how often an active molecule gets a higher score than an inactive one.
  0.5 is a coin toss, 1 is perfect.
- **PR-AUC**: how precise the model stays as it finds more of the actives. A coin toss
  scores the share of actives (here about 0.5), and 1 is perfect.
- **Balanced accuracy**: the average of the share of actives and the share of inactives
  it gets right.
- **Precision**: of the molecules it calls active, how many really are.
- **Recall**: of the real actives, how many it finds.

In [ ]:
pd.Series(modelling.classification_metrics(y_test, proba)).round(3).to_frame("test set")

The **ROC curve** shows the trade-off behind ROC-AUC. Moving along the curve lowers the
probability needed to call a molecule active: the model finds more actives (up) but
also calls more inactives active by mistake (right). The diagonal is a coin toss.

Next to it, the same predictions split by the real class. Each dot is one test
molecule, placed at the probability the model gave it, and the box covers the middle half
of each class. A good model pushes the actives up and the inactives down; the dots that
cross the dashed line at 0.5 are its mistakes.


In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, proba)
fig, axs = stylia.create_figure(1, 2, width=0.75, height=0.5, width_ratios=[2, 1])
ax = axs.next()
ax.plot(fpr, tpr, color=nc.purple)
ax.plot([0, 1], [0, 1], color=nc.gray, linestyle=":")
stylia.label(ax, xlabel="False positive rate", ylabel="True positive rate",
             title="ROC curve (test set)")
ax = axs.next()
modelling.plot_proba_by_class(ax, y_test, proba, colors=[nc.purple, nc.mint])
stylia.label(ax, xlabel="", ylabel="Predicted probability of being active",
             title="Scores by class")

The **confusion matrix** counts the four possible outcomes: actives called active,
actives missed, inactives called inactive, and inactives wrongly called active.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ConfusionMatrixDisplay.from_predictions(
    y_test, (proba >= 0.5).astype(int), display_labels=["inactive", "active"],
    cmap="Purples", colorbar=False, ax=ax)
ax.grid(False)
stylia.label(ax, xlabel="Predicted", ylabel="Real", title="Confusion matrix (test set)")

> **Exercise:** Change the 0.5 in the cell above to 0.3 and then to 0.7. How do the
> four numbers move? If the model is used to choose which natural products to buy and test,
> which mistake is more expensive: missing an active or testing an inactive?

### 5.2 Random or scaffold split: a fairer test

The split above was random. Because molecules come in series, a random split puts close
relatives of almost every test molecule into the training set. The model has then seen
something very similar before, and the test is easy.

That is not how the model will be used. The group wants to screen **African natural
products**, which look quite different from the synthetic compounds in ChEMBL. A fairer test
is a **scaffold split**: every molecule of a series goes to the same side, so the test
molecules have scaffolds the model has never seen.

We also switch from one split to **5-fold cross-validation**: the data is cut into five parts
(folds) and the model is trained five times, each time tested on a different fold. That gives
five scores instead of one, which shows how much the result depends on luck.

`StratifiedKFold` makes five random folds. `StratifiedGroupKFold` makes five folds where each
scaffold stays in a single fold. Both keep the share of actives even across folds. The cell
below trains ten models, so it takes a minute or two.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

random_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scaffold_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_class = pd.concat({
    "random": modelling.cross_validate(classifier, X_class, y_class, random_folds),
    "scaffold": modelling.cross_validate(classifier, X_class, y_class, scaffold_folds,
                                         groups=curated["scaffold"]),
}, names=["split"])
cv_class.groupby("split").agg(["mean", "std"]).round(3)

The scaffold split scores lower, and that lower number is the honest one: it is the
one to report and the one a better model has to beat.

## 6. Predict how active

Now the harder question: not just active or not, but **how** active. The model predicts the
pActivity itself, on the smaller table of molecules that have a real measured value.

### 6.1 A first model on a random split

We split 80/20 in the same way, without stratification this time, as there are no classes.

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_SEED)
print(f"training set: {len(yr_train):,} molecules")
print(f"test set:     {len(yr_test):,} molecules")

A random forest regressor works like the classifier, but each tree predicts a number
and the forest averages them. `max_features="sqrt"` lets each question in a tree
consider only a random handful of the 2,048 bits, which makes the trees more
varied and training much faster.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(n_estimators=100, max_features="sqrt",
                                  n_jobs=-1, random_state=RANDOM_SEED)
regressor.fit(Xr_train, yr_train)
predicted = regressor.predict(Xr_test)
print(f"predicted the pActivity of {len(predicted):,} test molecules")

The regression metrics mean:

- **R2** (R squared): how much of the spread in the real values the model explains. 0
  means no better than always guessing the average, 1 is perfect.
- **RMSE** and **MAE**: the typical error, in pActivity units. An error of 1 means the
  prediction is off by a factor of ten in concentration. RMSE punishes large mistakes
  more than MAE does.
- **Spearman**: whether the model puts the molecules in the right order, from least to
  most potent, even if the numbers themselves are off. 1 is a perfect ranking.

In [ ]:
pd.Series(modelling.regression_metrics(yr_test, predicted)).round(3).to_frame("test set")

Each point below is one test molecule. A perfect model would put every point on the
diagonal line.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ax.scatter(yr_test, predicted, color=nc.purple, alpha=0.3)
ax.plot([3, 11.5], [3, 11.5], color=nc.gray, linestyle=":")
stylia.label(ax, xlabel="Measured pActivity", ylabel="Predicted pActivity",
             title="Regression (test set)")

Notice how the points flatten out: very potent molecules are predicted too low and
very weak ones too high. Random forests average many trees, so they are pulled towards the
middle and rarely predict extreme values.

> **Note:** Remember from the curation notebook that repeated measurements of the same
> molecule often differ by about one log unit. A model cannot be more accurate than the data
> it learns from, so an error close to 1 may already be near the limit.

### 6.2 Random or scaffold split

The same comparison as for the classifier, with the same five scaffold groups.

In [ ]:
from sklearn.model_selection import GroupKFold, KFold

cv_reg = pd.concat({
    "random": modelling.cross_validate(
        regressor, X_reg, y_reg, KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)),
    "scaffold": modelling.cross_validate(
        regressor, X_reg, y_reg, GroupKFold(n_splits=5), groups=regression["scaffold"]),
}, names=["split"])
cv_reg.groupby("split").agg(["mean", "std"]).round(3)

Each dot below is one fold and the dark line is the mean over the five folds. The
further apart the two splits sit, the more the random split flattered the model.

In [ ]:
fig, axs = stylia.create_figure(1, 2)
for cv, metric in [(cv_class, "ROC-AUC"), (cv_reg, "R2")]:
    ax = axs.next()
    for i, (split, color) in enumerate([("random", "purple"), ("scaffold", "mint")]):
        scores = cv.loc[split, metric]
        ax.scatter([i] * len(scores), scores, color=nc.get(color))
        ax.hlines(scores.mean(), i - 0.25, i + 0.25, color=nc.plum)
    ax.set_xticks([0, 1], ["random", "scaffold"])
    ax.set_xlim(-0.6, 1.6)
    stylia.label(ax, xlabel="Split", ylabel=metric, title=f"{metric} over 5 folds")

Both models score worse on the scaffold split, and the regressor loses more. The
scaffold numbers are the ones to report, because they are closer to what happens when the
model meets molecules from a new chemical series, like natural products.

> **Exercise:** Change the fingerprint in section 4, for example to `radius=3` or
> `n_bits=1024`, and run the notebook again from there. Does the scaffold split score change?
> Try also removing `class_weight="balanced"` from the classifier.

## 7. Train the final models and save them

Everything so far was measurement: each model was trained on part of the data so that the
rest could test it. Now that we know roughly how well the method works, we train the two
final models on **all** the data, because more training data makes a better model.

A final model has no test score of its own, since nothing is left to test it on. The numbers
that belong with it are the cross-validation scores from sections 5 and 6, so write those
down next to the file.

In [ ]:
final_classifier = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                          n_jobs=-1, random_state=RANDOM_SEED)
final_classifier.fit(X_class, y_class)

final_regressor = RandomForestRegressor(n_estimators=100, max_features="sqrt",
                                        n_jobs=-1, random_state=RANDOM_SEED)
final_regressor.fit(X_reg, y_reg)
print(f"classifier trained on {len(y_class):,} molecules, "
      f"regressor on {len(y_reg):,}")

`save_model` writes the model into `outputs/` and, in Colab, downloads it to your
computer. Do keep it: a Colab runtime is deleted when it disconnects, and the file in
`outputs/` goes with it. The download is the copy that lasts, and it belongs in the group's
Drive folder afterwards.

In [ ]:
modelling.save_model(final_classifier, "purple_activity_classifier.joblib")
modelling.save_model(final_regressor, "purple_pactivity_regressor.joblib")

To use a saved model later, load it and give it fingerprints made exactly the same
way, with `radius=2` and `n_bits=2048`. A model and its featurisation belong together: the
same molecule described differently is a different row of numbers, and the predictions would
be meaningless.

In [ ]:
import joblib

reloaded = joblib.load("outputs/purple_activity_classifier.joblib")
examples = ["CC(C)(C)NC(=O)C1CC2CCCCC2CN1CC(O)C(Cc1ccccc1)NC(=O)C(CC(N)=O)NC(=O)c1ccc2ccccc2n1",
            "CCO"]
probability = reloaded.predict_proba(modelling.morgan_fingerprints(examples))[:, 1]
pd.DataFrame({"molecule": ["indinavir (an HIV drug)", "ethanol"],
              "probability of being active": probability.round(3)})

The model is confident about the drug and not about the solvent, which is the least we
should ask of it. Be careful how much comfort you take from this, though: indinavir is one
of the molecules the model was trained on, so this shows how to call the model, not how well
it works. The honest numbers are the scaffold-split ones from sections 5 and 6.

> **Exercise:** Add a few molecules of your own to the list, for example natural products
> the group is interested in, and see what the model says. Remember that a molecule far from
> anything in the training set gets a prediction the model has no real basis for.

## Summary

- You looked at the data, then removed the molecules above 1,000 g/mol: a small share of the
  table, and a part of chemical space the group will never screen.
- You turned each molecule into a 2,048-bit Morgan fingerprint and trained a random forest
  classifier (active or not) and a random forest regressor (pActivity).
- Both were scored twice. On a random split the classifier reaches a ROC-AUC of about 0.93
  and the regressor an R2 of about 0.71; on a scaffold split they fall to about 0.89 and
  0.59. The scaffold numbers are the fair ones for molecules from new series.
- You trained both final models on all the data and downloaded them, with the
  cross-validation scores as the numbers to quote alongside.

**Next:** use the classifier to score a library of African natural products, and compare
what it picks with the chemical space map from `purple_chemical_space`.